# BERT Model

Questo notebook analizza le descrizioni testuali degli annunci di adozione utilizzando BERT.

L'obiettivo è verificare se il contenuto testuale delle descrizioni può contribuire alla previsione della variabile `AdoptionSpeed`.

Rispetto al notebook `02_baseline_model.ipynb`, che lavora principalmente su dati strutturati e razze, qui il focus è sulla componente testuale.

Alla fine viene aggiornato anche il confronto tra modelli, includendo il risultato della Random Forest con razza.


## 1. Import delle librerie

In [1]:
import pandas as pd
import numpy as np
import torch

from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

## 2. Caricamento del dataset

Viene caricato il dataset pulito `dogs_clean.csv`.

Il notebook utilizza principalmente:

- `Description`: descrizione testuale dell'annuncio;
- `AdoptionSpeed`: variabile target.


In [2]:
dogs = pd.read_csv("../data/processed/dogs_clean.csv")

texts = dogs["Description"].fillna("").tolist()
y = dogs["AdoptionSpeed"]

print("Dataset:", dogs.shape)
print("Numero descrizioni:", len(texts))

dogs[["Name", "Description", "AdoptionSpeed"]].head()

Dataset: (8132, 32)
Numero descrizioni: 8132


,Name,Description,AdoptionSpeed
0,Brisco,Their pregnant mother was dumped by her irresp...,3
1,Miko,"Good guard dog, very alert, active, obedience ...",2
2,Hunter,This handsome yet cute boy is up for adoption....,2
3,Siu Pak & Her 6 Puppies,Siu Pak just give birth on 13/6/10 to 6puppies...,3
4,Bear,"For serious adopter, please do sms or call for...",1


## 3. Caricamento di BERT

Viene utilizzato il modello pre-addestrato `bert-base-uncased`.

BERT trasforma ogni descrizione testuale in un vettore numerico, detto embedding.  
Ogni embedding ha dimensione 768.


In [3]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)

bert_model.eval()

print("BERT model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model loaded.


## 4. Funzione per estrarre embedding BERT

La funzione seguente:

1. tokenizza il testo;
2. lo passa a BERT;
3. calcola la media degli embedding dei token;
4. restituisce un vettore numerico di dimensione 768.


In [4]:
def get_bert_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = bert_model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.squeeze().numpy()

## 5. Caricamento o creazione degli embedding

Gli embedding BERT vengono salvati su file per evitare di ricalcolarli a ogni esecuzione.

Se il file `bert_embeddings.npy` esiste già, viene semplicemente caricato.  
In caso contrario, gli embedding vengono calcolati e salvati.


In [5]:
try:
    bert_embeddings = np.load("../data/processed/bert_embeddings.npy")
    print("Embedding caricati:", bert_embeddings.shape)

except FileNotFoundError:
    bert_embeddings = []

    for text in tqdm(texts):
        emb = get_bert_embedding(text)
        bert_embeddings.append(emb)

    bert_embeddings = np.array(bert_embeddings)
    np.save("../data/processed/bert_embeddings.npy", bert_embeddings)

    print("Embedding creati e salvati:", bert_embeddings.shape)

Embedding caricati: (8132, 768)


## 6. Modello BERT + Logistic Regression

Gli embedding BERT vengono usati come input per una Logistic Regression.

In questo esperimento il modello usa **solo le descrizioni testuali**, senza dati strutturati come età, taglia, razza o numero di fotografie.


In [6]:
X = bert_embeddings

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

bert_clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

bert_clf.fit(X_train, y_train)

y_pred_bert = bert_clf.predict(X_test)

bert_accuracy = accuracy_score(y_test, y_pred_bert)

print("Accuracy BERT:", bert_accuracy)
print(classification_report(y_test, y_pred_bert))

Accuracy BERT: 0.3306699446834665
              precision    recall  f1-score   support

           0       0.02      0.09      0.04        34
           1       0.28      0.34      0.31       287
           2       0.36      0.28      0.31       433
           3       0.33      0.33      0.33       390
           4       0.45      0.38      0.41       483

    accuracy                           0.33      1627
   macro avg       0.29      0.29      0.28      1627
weighted avg       0.36      0.33      0.34      1627



## 7. Modello combinato: feature tabellari + BERT

In questo esperimento vengono unite:

- feature strutturate del cane;
- embedding BERT della descrizione.

L'obiettivo è verificare se la combinazione tra dati tabellari e testo migliora le prestazioni.

Nota: in questa versione il modello combinato usa le feature tabellari principali, ma non include il one-hot encoding della razza. Il ruolo della razza è stato analizzato nel notebook `02_baseline_model.ipynb`.


In [7]:
tabular_features = [
    "Age",
    "Gender",
    "MaturitySize",
    "FurLength",
    "Vaccinated",
    "Dewormed",
    "Sterilized",
    "Health",
    "Quantity",
    "Fee",
    "PhotoAmt",
    "description_len",
    "has_description"
]

X_tabular = dogs[tabular_features].values

X_combined = np.hstack([
    X_tabular,
    bert_embeddings
])

print("Feature tabellari:", X_tabular.shape)
print("Feature combinate:", X_combined.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X_combined,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

combined_clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

combined_clf.fit(X_train, y_train)

y_pred_combined = combined_clf.predict(X_test)

combined_accuracy = accuracy_score(y_test, y_pred_combined)

print("Accuracy modello combinato:", combined_accuracy)
print(classification_report(y_test, y_pred_combined))

Feature tabellari: (8132, 13)
Feature combinate: (8132, 781)
Accuracy modello combinato: 0.31161647203441917
              precision    recall  f1-score   support

           0       0.04      0.24      0.06        34
           1       0.25      0.28      0.26       287
           2       0.32      0.20      0.24       433
           3       0.34      0.34      0.34       390
           4       0.47      0.41      0.44       483

    accuracy                           0.31      1627
   macro avg       0.28      0.29      0.27      1627
weighted avg       0.35      0.31      0.33      1627



c:\Users\sofia\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 8. Confronto finale tra i modelli

Il confronto include:

- Random Forest senza razza;
- Random Forest con razza;
- BERT + Logistic Regression;
- modello combinato tabellare + BERT.

Dopo l'aggiunta della razza, la Random Forest ha migliorato le prestazioni, passando da circa 40.93% a circa 44.13%.


In [8]:
results = pd.DataFrame({
    "Model": [
        "Random Forest senza razza",
        "Random Forest con razza",
        "BERT + Logistic Regression",
        "Tabellare + BERT"
    ],
    "Accuracy": [
        0.4093,
        0.4413,
        bert_accuracy,
        combined_accuracy
    ]
})

results

,Model,Accuracy
0,Random Forest senza razza,0.409300
1,Random Forest con razza,0.441300
2,BERT + Logistic Regression,0.330670
3,Tabellare + BERT,0.311616


## 9. Conclusioni

Il modello basato esclusivamente su BERT ha ottenuto un'accuracy inferiore rispetto alla Random Forest basata su dati strutturati.

Questo indica che le sole descrizioni testuali non sono sufficienti per superare le informazioni strutturate del dataset.

L'aggiunta della razza nel notebook `02_baseline_model.ipynb` ha migliorato ulteriormente la Random Forest, portando l'accuracy a circa 44.13%. Questo suggerisce che la razza contiene informazione utile per prevedere la velocità di adozione.

BERT rimane comunque importante nel progetto perché viene usato nel sistema di matching cane–famiglia per confrontare semanticamente:

- descrizione del cane ideale inserita dalla famiglia;
- descrizione dell'annuncio del cane.

In questo modo la componente testuale contribuisce alla raccomandazione personalizzata, anche se non è il miglior modello predittivo per `AdoptionSpeed`.
